In [ ]:
import copy
from itertools import product as cartesian_product

class AutomatedWarehouseRobotControllerProblem:
    """Encapsulates the full MAPF problem for the warehouse robot fleet"""

    # The 5 possible moves per robot: Right, Down, Left, Up, Wait
    # (dx, dy) format — added to current (x, y) to get next position
    MOVES = [(0, 1), (1, 0), (0, -1), (-1, 0), (0, 0)]

    # Human-readable names for each move, used in debugging/logging
    MOVE_NAMES = {
        (0,  1): 'Right',
        (1,  0): 'Down',
        (0, -1): 'Left',
        (-1, 0): 'Up',
        (0,  0): 'Wait',
    }

    def __init__(self, initial_state, robots, max_time=500):
        """Initialize the MAPF problem

        Args:
            initial_state (dict): Fully-populated state dict:
                {
                    'robots'    : [ {id, start_position, goal_position, color, at_goal}, ... ],
                    'positions' : { robot_id: (x, y), ... },
                    'collisions': 0,
                    'deadlock'  : False,
                    'timestep'  : 0,
                    'grid'      : GridEnvironment
                }
            robots   (list): List of Robot objects
            max_time (int) : Hard cap on timesteps to prevent infinite loops
        """
        # deep copy so the original state is never mutated by the search
        self.initial_state = copy.deepcopy(initial_state)

        # list of Robot objects — used for reset() and move_to() mirroring
        self.robots     = robots

        # dict {robot_id → Robot} for O(1) lookup instead of linear scan
        self.robots_map = {r.robot_id: r for r in robots}

        # total number of robots in this problem instance
        self.num_robots = len(robots)

        # reference to the shared GridEnvironment
        self.grid = initial_state['grid']

        # maximum allowed timesteps before we declare failure
        # prevents A* from running forever on unsolvable instances
        self.max_time = max_time

        # ---- Cooperative A* reservation table ----
        # key: (x, y, t) → value: robot_id
        # when robot R plans its path, it writes every (x, y, t) it will occupy
        # the next robot then treats those cells as blocked at those times
        self.reservation_tbl = {}

        # ---- Conflict log ----
        # every conflict detected by apply_action is appended here
        # used later by get_congestion_heatmap() to show which corridors are hotspots
        self.conflict_log = []

        # ---- Aggregate statistics ----
        # updated automatically during the search, read by the evaluation section
        self.stats = {
            'total_vertex_conflicts': 0,  # how many vertex collisions occurred
            'total_edge_conflicts'  : 0,  # how many swap collisions occurred
            'total_deadlocks'       : 0,  # how many times deadlock was detected
            'total_steps'           : 0,  # total moves made across all robots
            'nodes_expanded'        : 0,  # how many nodes A* expanded
        }

    # ================================================================== #
    #  GOAL TEST                                                           #
    # ================================================================== #

    def is_goal(self, state):
        """Check if the given state is a goal state

        A state is a goal when:
          1. Every robot has reached its goal position
          2. No collisions occurred
          3. No deadlock is present

        Args:
            state (dict): Current state

        Returns:
            bool: True if goal state
        """
        # immediately reject states with any collision or deadlock
        if state['collisions'] > 0 or state['deadlock']:
            return False

        # every single robot must have at_goal = True
        return all(robot['at_goal'] for robot in state['robots'])

    def is_partial_goal(self, state, robot_id):
        """Check if a single specific robot has reached its goal.
        Used by Cooperative A* which plans one robot at a time.

        Args:
            state    (dict): Current state
            robot_id (int) : The robot to check

        Returns:
            bool: True if that robot is at its goal
        """
        pos   = state['positions'][robot_id]
        robot = next(r for r in state['robots'] if r['id'] == robot_id)
        return pos == tuple(robot['goal_position'])

    # ================================================================== #
    #  VALID ACTIONS                                                       #
    # ================================================================== #

    def get_valid_actions(self, state, robot_ids=None, use_reservation=False):
        """Generate all valid joint actions for the given state.

        A joint action is one move decision per robot simultaneously.
        e.g. {0: (3,4), 1: (7,2), 2: (1,1)} means:
             robot 0 moves to (3,4), robot 1 to (7,2), robot 2 to (1,1)

        Args:
            state           (dict): Current state
            robot_ids       (list): Subset of robot ids to plan for.
                                    None = plan for all robots.
                                    Cooperative A* passes one robot at a time.
            use_reservation (bool): If True, skip cells reserved by other robots.
                                    Used in Cooperative A* mode.

        Returns:
            list[dict]: List of joint-action dicts [{robot_id: (nx, ny), ...}, ...]
        """
        grid        = state['grid']
        timestep    = state.get('timestep', 0)

        # decide which robots we're planning for
        ids_to_plan = robot_ids if robot_ids is not None else [r['id'] for r in state['robots']]

        # build a list of candidate next-positions for each robot
        per_robot_options = []

        for rid in ids_to_plan:
            x, y        = state['positions'][rid]
            robot_entry = next(r for r in state['robots'] if r['id'] == rid)

            # a robot already at its goal should just wait there
            # this keeps it out of the way of other robots still moving
            if robot_entry['at_goal']:
                per_robot_options.append([(x, y)])
                continue

            candidates = []
            for dx, dy in self.MOVES:
                nx, ny = x + dx, y + dy

                # skip cells that are walls or outside grid bounds
                if not grid.is_walkable(nx, ny):
                    continue

                # Cooperative A* reservation check:
                # if another robot has already reserved (nx, ny) at time t+1, skip it
                if use_reservation:
                    reserved_by = self.reservation_tbl.get((nx, ny, timestep + 1))
                    if reserved_by is not None and reserved_by != rid:
                        continue

                    # also block swap conflicts via reservation:
                    # if robot B is at (nx, ny) now and moving to (x, y) next step
                    # that would be a head-on swap → skip
                    swap_reserved = self.reservation_tbl.get((x, y, timestep + 1))
                    if swap_reserved is not None and swap_reserved != rid:
                        if self.reservation_tbl.get((nx, ny, timestep)) == swap_reserved:
                            continue

                candidates.append((nx, ny))

            # if no moves available, the robot must wait in place
            if not candidates:
                candidates = [(x, y)]

            per_robot_options.append(candidates)

        # Cartesian product: combine one option per robot → all possible joint actions
        # e.g. robot0 has 3 options, robot1 has 2 → 6 joint actions total
        joint_actions = []
        for combo in cartesian_product(*per_robot_options):
            joint_action = {rid: pos for rid, pos in zip(ids_to_plan, combo)}
            joint_actions.append(joint_action)

        return joint_actions

    # ================================================================== #
    #  CONFLICT DETECTION                                                  #
    # ================================================================== #

    def detect_vertex_collision(self, action):
        """Detect vertex conflicts: two robots moving to the same cell at time t+1.

        Example: robot A moves to (3,4) AND robot B moves to (3,4) → conflict

        Args:
            action (dict): Joint action {robot_id: (nx, ny)}

        Returns:
            list[dict]: List of vertex conflict dicts, empty if none
        """
        # group robots by their destination cell
        pos_to_robots = {}
        for rid, pos in action.items():
            pos_to_robots.setdefault(pos, []).append(rid)

        conflicts = []
        for pos, rids in pos_to_robots.items():
            if len(rids) > 1:
                # every pair of robots going to the same cell is a separate conflict
                for i in range(len(rids)):
                    for j in range(i + 1, len(rids)):
                        conflicts.append({
                            'type'    : 'vertex',
                            'robots'  : [rids[i], rids[j]],
                            'location': pos,
                            'timestep': None  # filled in by apply_action
                        })
        return conflicts

    def detect_edge_collision(self, state, action):
        """Detect edge (swap) conflicts: two robots swapping positions simultaneously.

        Example: robot A moves (1,2)→(1,3) while robot B moves (1,3)→(1,2)
                 they pass through each other → conflict

        Args:
            state  (dict): Current state (to get current positions)
            action (dict): Joint action {robot_id: (nx, ny)}

        Returns:
            list[dict]: List of edge conflict dicts, empty if none
        """
        current   = state['positions']
        rids      = list(action.keys())
        conflicts = []

        for i in range(len(rids)):
            for j in range(i + 1, len(rids)):
                ra, rb       = rids[i], rids[j]
                from_a, to_a = current[ra], action[ra]
                from_b, to_b = current[rb], action[rb]

                # swap conflict: A goes from_a→to_a = from_b, B goes from_b→to_b = from_a
                if to_a == from_b and to_b == from_a:
                    conflicts.append({
                        'type'    : 'edge',
                        'robots'  : [ra, rb],
                        'location': (from_a, from_b),  # the two cells involved
                        'timestep': None
                    })
        return conflicts

    def detect_deadlock(self, state):
        """Detect circular-wait deadlocks using DFS cycle detection.

        A deadlock occurs when a group of robots are each waiting for
        the next robot in a cycle to move first — nobody can move.

        Example: A waits for B, B waits for C, C waits for A → deadlock

        Args:
            state (dict): Current state

        Returns:
            bool: True if a deadlock cycle exists
        """
        positions  = state['positions']
        # reverse map: (x,y) → robot_id, so we can check who occupies a cell
        pos_to_rid = {v: k for k, v in positions.items()}
        grid       = state['grid']

        # build "blocked_by" graph: robot → set of robots blocking its neighbours
        # robot R is blocked by robot S if S sits on one of R's walkable neighbours
        blocked_by = {}
        for robot in state['robots']:
            rid        = robot['id']
            x, y       = positions[rid]
            neighbours = grid.get_neighbors(x, y)
            blockers   = {
                pos_to_rid[n] for n in neighbours
                if n in pos_to_rid and pos_to_rid[n] != rid
            }
            blocked_by[rid] = blockers

        # standard DFS cycle detection
        # visited   : nodes we've fully explored (no cycle through them)
        # rec_stack : nodes on the current DFS path (cycle = revisiting one of these)
        visited, rec_stack = set(), set()

        def has_cycle(node):
            visited.add(node)
            rec_stack.add(node)
            for neighbour in blocked_by.get(node, []):
                if neighbour not in visited:
                    if has_cycle(neighbour):
                        return True
                elif neighbour in rec_stack:
                    # found a back-edge → cycle exists
                    return True
            rec_stack.discard(node)
            return False

        return any(has_cycle(rid) for rid in blocked_by if rid not in visited)

    def count_conflicts(self, state, action):
        """Count total conflicts (vertex + edge) for a joint action.
        Used by Hill Climbing to score and compare candidate moves.

        Args:
            state  (dict): Current state
            action (dict): Joint action to evaluate

        Returns:
            int: Total number of conflicts
        """
        return (len(self.detect_vertex_collision(action)) +
                len(self.detect_edge_collision(state, action)))

    # ================================================================== #
    #  STATE TRANSITION                                                    #
    # ================================================================== #

    def apply_action(self, state, action):
        """Apply a joint action to a state and return the resulting new state.

        This is the transition model: state + action → new_state

        What it updates:
          - positions of all robots
          - each robot's at_goal, status, steps_taken, waits, arrival_time
          - collision count
          - deadlock flag
          - timestep counter
          - conflict_log and stats (side effects for analysis/heatmap)

        Args:
            state  (dict): Current state (not mutated)
            action (dict): Joint action {robot_id: (nx, ny)}

        Returns:
            dict: New state (deep copy)
        """
        # deep copy so the original state is never mutated
        # this is critical for tree search — parent nodes must stay unchanged
        new_state = copy.deepcopy(state)
        timestep  = new_state.get('timestep', 0) + 1
        new_state['timestep'] = timestep

        # --- detect conflicts BEFORE applying moves ---
        # we check against the OLD positions (current state), not the new ones
        v_conflicts = self.detect_vertex_collision(action)
        e_conflicts = self.detect_edge_collision(state, action)

        # log every conflict and increment the collision counter
        for c in v_conflicts + e_conflicts:
            c['timestep'] = timestep
            self.conflict_log.append(c)       # saved for heatmap generation
            new_state['collisions'] += 1

        # update aggregate stats
        self.stats['total_vertex_conflicts'] += len(v_conflicts)
        self.stats['total_edge_conflicts']   += len(e_conflicts)

        # --- apply moves to each robot ---
        for robot_entry in new_state['robots']:
            rid     = robot_entry['id']
            new_pos = action[rid]
            old_pos = new_state['positions'][rid]

            # update the joint positions dict
            new_state['positions'][rid] = new_pos

            # update per-robot stats stored inside the state dict
            if new_pos == old_pos:
                # robot stayed in place → count as a wait
                robot_entry['waits']  = robot_entry.get('waits', 0) + 1
                robot_entry['status'] = 'waiting'
            else:
                # robot actually moved → count as a step
                robot_entry['steps_taken'] = robot_entry.get('steps_taken', 0) + 1
                robot_entry['status']      = 'moving'
                self.stats['total_steps'] += 1

            # check if robot just reached its goal
            at_goal = (new_pos == tuple(robot_entry['goal_position']))
            robot_entry['at_goal'] = at_goal

            if at_goal:
                robot_entry['status'] = 'done'
                # only record arrival_time the FIRST time it reaches the goal
                if robot_entry.get('arrival_time', -1) == -1:
                    robot_entry['arrival_time'] = timestep

            # mirror the move into the Robot object (keeps Robot in sync with state)
            if rid in self.robots_map:
                self.robots_map[rid].move_to(new_pos, timestep)

        # --- check for deadlock in the new state ---
        deadlock = self.detect_deadlock(new_state)
        new_state['deadlock'] = deadlock
        if deadlock:
            self.stats['total_deadlocks'] += 1

        return new_state

    # ================================================================== #
    #  NODE EXPANSION                                                      #
    # ================================================================== #

    def expand_node(self, node, use_reservation=False):
        """Generate all child Nodes from the given node.

        Called by A* at every expansion step.
        Each child = one valid joint action applied to the current node's state.
        g increases by 1 (one timestep).
        h is recomputed using the Manhattan distance heuristic.

        Args:
            node            (Node): Node to expand
            use_reservation (bool): True for Cooperative A* mode

        Returns:
            list[Node]: Child nodes
        """
        # track how many nodes the search has expanded (for performance comparison)
        self.stats['nodes_expanded'] += 1
        children = []

        for action in self.get_valid_actions(node.state, use_reservation=use_reservation):
            new_state = self.apply_action(node.state, action)
            g         = node.g + 1          # one more timestep
            h         = self.heuristic(new_state)
            child     = Node(state=new_state, parent=node, action=action, g=g, h=h)
            children.append(child)

        return children

    # ================================================================== #
    #  HEURISTIC                                                           #
    # ================================================================== #

    def heuristic(self, state):
        """Compute the admissible heuristic for a state.

        Uses sum of Manhattan distances from each robot's current position
        to its goal position.

        Admissible because Manhattan distance never overestimates:
        a robot needs AT LEAST |Δx| + |Δy| steps to reach its goal
        even with no obstacles.

        Robots already at their goal contribute 0.

        Args:
            state (dict): Current state

        Returns:
            float: Heuristic value (lower = closer to goal)
        """
        total = 0
        for robot in state['robots']:
            if robot['at_goal']:
                continue  # already done, no cost to add
            rid    = robot['id']
            cx, cy = state['positions'][rid]
            gx, gy = robot['goal_position']
            total += abs(cx - gx) + abs(cy - gy)
        return total

    # ================================================================== #
    #  METRICS                                                             #
    # ================================================================== #

    def get_makespan(self, paths):
        """Compute makespan: time when the LAST robot reaches its goal.

        Minimizing makespan = getting everyone done as fast as possible.

        Args:
            paths (dict): {robot_id: [(x0,y0), ..., (xT,yT)]}

        Returns:
            int: Maximum path length across all robots
        """
        if not paths:
            return 0
        return max(len(p) - 1 for p in paths.values())

    def get_flowtime(self, paths, state):
        """Compute flowtime: sum of individual arrival times across all robots.

        Minimizing flowtime = minimizing total time robots spend travelling.
        A robot that arrives early contributes a small number.

        Args:
            paths (dict): {robot_id: [(x0,y0), ..., (xT,yT)]}
            state (dict): Final state (to look up goal positions)

        Returns:
            int: Sum of arrival timesteps
        """
        goals = {r['id']: tuple(r['goal_position']) for r in state['robots']}
        total = 0
        for rid, path in paths.items():
            goal = goals[rid]
            for t, pos in enumerate(path):
                if pos == goal:
                    total += t
                    break
            else:
                # robot never reached goal → penalise with full path length
                total += len(path) - 1
        return total

    def get_per_robot_stats(self, state):
        """Return per-robot breakdown of steps, waits, and arrival time.

        Used by the comparative evaluation section to analyse
        individual robot performance across different algorithms.

        Args:
            state (dict): Final state

        Returns:
            dict: {robot_id: {'steps_taken': int, 'waits': int, 'arrival_time': int}}
        """
        result = {}
        for robot in state['robots']:
            result[robot['id']] = {
                'steps_taken' : robot.get('steps_taken', 0),
                'waits'       : robot.get('waits', 0),
                'arrival_time': robot.get('arrival_time', -1),
            }
        return result

    # ================================================================== #
    #  RESERVATION TABLE (Cooperative A*)                                 #
    # ================================================================== #

    def reserve_path(self, robot_id, path):
        """Add a robot's planned path to the reservation table.

        Called after each robot finishes planning in Cooperative A*.
        The next robot will see these reservations and avoid those cells
        at those timesteps, preventing conflicts before they happen.

        Args:
            robot_id (int) : Robot whose path we're locking in
            path     (list): [(x0,y0), ..., (xT,yT)]
        """
        for t, (x, y) in enumerate(path):
            # key = (x, y, timestep) → value = robot_id who owns this cell at time t
            self.reservation_tbl[(x, y, t)] = robot_id

    def clear_reservations(self):
        """Wipe the reservation table.
        Call this before re-planning the entire fleet from scratch.
        """
        self.reservation_tbl.clear()

    def is_cell_reserved(self, x, y, t, asking_robot):
        """Check if a cell is reserved by someone other than the asking robot.

        Args:
            x            (int): Cell x coordinate
            y            (int): Cell y coordinate
            t            (int): Timestep to check
            asking_robot (int): Robot id doing the asking

        Returns:
            bool: True if reserved by a different robot
        """
        owner = self.reservation_tbl.get((x, y, t))
        return owner is not None and owner != asking_robot

    # ================================================================== #
    #  HEATMAP                                                             #
    # ================================================================== #

    def get_congestion_heatmap(self):
        """Build a cell-level congestion map from the conflict log.

        Counts how many conflicts happened at each grid cell.
        High counts = bottleneck corridors.
        Used to generate the heatmap deliverable.

        Returns:
            dict: {(x, y): conflict_count}
        """
        heatmap = {}
        for conflict in self.conflict_log:
            loc = conflict['location']
            if conflict['type'] == 'vertex':
                # one cell involved
                heatmap[loc] = heatmap.get(loc, 0) + 1
            else:
                # edge conflict: two cells involved (the swap corridor)
                for cell in loc:
                    heatmap[cell] = heatmap.get(cell, 0) + 1
        return heatmap

    # ================================================================== #
    #  LOCAL SEARCH SUPPORT                                                #
    # ================================================================== #

    def generate_neighbors(self, state):
        """Generate neighboring states for Hill Climbing.

        Each neighbor = apply one valid joint action to the current state.
        Hill Climbing picks the neighbor with the best evaluation score.

        Args:
            state (dict): Current state

        Returns:
            list[tuple]: [(action, new_state), ...]
        """
        return [
            (action, self.apply_action(state, action))
            for action in self.get_valid_actions(state)
        ]

    def evaluate(self, state, paths):
        """Compute all objective metrics for a solution in one call.

        Used by Hill Climbing and the comparative evaluation section.

        Args:
            state (dict): Final state
            paths (dict): {robot_id: path}

        Returns:
            dict: {
                'makespan'  : int,   ← time until last robot done
                'flowtime'  : int,   ← sum of individual travel times
                'collisions': int,   ← total collisions that occurred
                'deadlock'  : bool,  ← whether a deadlock was detected
                'success'   : bool   ← True if all robots reached goals cleanly
            }
        """
        return {
            'makespan'  : self.get_makespan(paths),
            'flowtime'  : self.get_flowtime(paths, state),
            'collisions': state['collisions'],
            'deadlock'  : state['deadlock'],
            'success'   : self.is_goal(state),
        }

    # ================================================================== #
    #  PATH EXTRACTION                                                     #
    # ================================================================== #

    def extract_paths(self, goal_node):
        """Reconstruct each robot's full path by tracing parent pointers.

        Walks from the goal node back to the root, collecting positions
        at each timestep, then reverses the result.

        Args:
            goal_node (Node): The goal node found by A*

        Returns:
            dict: {robot_id: [(x0,y0), (x1,y1), ..., (xT,yT)]}
        """
        # get ordered list of nodes from root → goal
        nodes = goal_node.get_path_to_root()
        rids  = [r['id'] for r in self.initial_state['robots']]

        # for each timestep (node), record where every robot was
        paths = {rid: [] for rid in rids}
        for n in nodes:
            for rid in rids:
                paths[rid].append(n.state['positions'][rid])
        return paths

    # ================================================================== #
    #  UTILITY                                                             #
    # ================================================================== #

    def reset(self):
        """Reset the problem back to its initial state.

        Clears all stats, reservations, and conflict logs.
        Resets all Robot objects to their start positions.
        Call this between algorithm runs to get a clean comparison.
        """
        self.reservation_tbl.clear()
        self.conflict_log.clear()
        self.stats = {k: 0 for k in self.stats}
        for robot in self.robots:
            robot.reset()

    def __repr__(self):
        return (f"AutomatedWarehouseRobotControllerProblem("
                f"robots={self.num_robots}, "
                f"grid={self.grid.width}x{self.grid.height})")

print("Class AutomatedWarehouseRobotControllerProblem defined")